# EDA & Error Analysis — Predicting Smartphone Addiction

This notebook documents the exploratory data analysis and post-modeling
error analysis for the Kaggle Playground Series S6E8 competition.
Modeling, tuning, and ensembling code lives in `src/`; this notebook
focuses on understanding the data and diagnosing model behavior.

**Dataset:** 691,369 train rows / 296,302 test rows, 12 behavioral features,
binary target `addicted_label` (70.9% / 29.1% class split). Evaluated on ROC-AUC.

## 1. Exploratory Data Analysis

Goal: understand the target distribution, diagnose the missing-data
mechanism (all 12 features have 4–19% missingness), and identify
predictive signal before deciding on a cleaning/preprocessing strategy.

Key question this section answers: **is missingness related to the
target (MNAR), or unrelated (MCAR/MAR)?** This determines whether
missing values should be imputed, encoded as a separate category, or
left as-is for tree-based models to handle natively.

In [ ]:
# Load data into pandas
TRAIN_PATH = "/kaggle/input/<competition-slug>/train.csv"
TEST_PATH = "/kaggle/input/<competition-slug>/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [ ]:
# Overview of train set
#train_df.head()
#train_df.describe()
#train_df.describe(include='object')
#train_df.info()
#train_df.shape
#train_df.isnull().sum()

# Overview of test set
#test_df.info()
#test_df.isnull().sum()
#test_df.shape

In [ ]:
from scipy import stats
import numpy as np 
import pandas as pd 

TARGET = "addicted_label"
ID_COL = "id"

numeric_cols = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
categorical_cols = ["gender", "stress_level", "academic_work_impact"]


# -----------------------------
# 1. Target class balance
# -----------------------------
def report_target_balance(df: pd.DataFrame, target_col: str) -> None:
    counts = df[target_col].value_counts()
    ratios = df[target_col].value_counts(normalize=True)
    print("=== Target class balance ===")
    print(pd.DataFrame({"count": counts, "ratio": ratios}))
    print()


# -----------------------------
# 2. Missing value pattern vs target
# -----------------------------
def report_missing_vs_target(df: pd.DataFrame, cols: list[str], target_col: str) -> pd.DataFrame:
    """
    For each column, compare addicted_label rate between rows where
    the column is missing vs not missing. Large gap => missing might be
    informative (MNAR-like), not just random noise.
    """
    results = []
    overall_rate = df[target_col].mean()

    for col in cols:
        is_missing = df[col].isna()
        rate_missing = df.loc[is_missing, target_col].mean() if is_missing.sum() > 0 else np.nan
        rate_not_missing = df.loc[~is_missing, target_col].mean()
        results.append({
            "column": col,
            "missing_count": is_missing.sum(),
            "missing_pct": is_missing.mean() * 100,
            "target_rate_when_missing": rate_missing,
            "target_rate_when_present": rate_not_missing,
            "diff_vs_overall": (rate_missing - overall_rate) if not np.isnan(rate_missing) else np.nan,
        })

    result_df = pd.DataFrame(results).sort_values("missing_pct", ascending=False)
    print("=== Missing value pattern vs target ===")
    print(f"Overall target rate: {overall_rate:.4f}")
    print(result_df.to_string(index=False))
    print()
    return result_df


# -----------------------------
# 3. Missing correlation between columns (co-occurrence)
# -----------------------------
def report_missing_cooccurrence(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    missing_matrix = df[cols].isna().astype(int)
    corr = missing_matrix.corr()
    print("=== Missing co-occurrence correlation (which columns tend to be missing together) ===")
    print(corr.round(2))
    print()
    return corr


# -----------------------------
# 4. Numeric distribution summary (skew, outlier count via IQR)
# -----------------------------
def report_numeric_distribution(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    results = []
    for col in cols:
        series = df[col].dropna()
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()

        results.append({
            "column": col,
            "mean": series.mean(),
            "median": series.median(),
            "std": series.std(),
            "skew": series.skew(),
            "min": series.min(),
            "max": series.max(),
            "iqr_outlier_count": outlier_count,
            "iqr_outlier_pct": outlier_count / len(series) * 100,
        })

    result_df = pd.DataFrame(results)
    print("=== Numeric distribution summary ===")
    print(result_df.round(3).to_string(index=False))
    print()
    return result_df


# -----------------------------
# 5. Correlation with target + multicollinearity
# -----------------------------
def report_numeric_correlation(df: pd.DataFrame, cols: list[str], target_col: str) -> None:
    corr_with_target = df[cols + [target_col]].corr()[target_col].drop(target_col)
    print("=== Correlation with target ===")
    print(corr_with_target.sort_values(ascending=False).round(3))
    print()

    print("=== Feature-feature correlation matrix (check multicollinearity) ===")
    print(df[cols].corr().round(2))
    print()


# -----------------------------
# 6. Categorical breakdown vs target
# -----------------------------
def report_categorical_vs_target(df: pd.DataFrame, cols: list[str], target_col: str) -> None:
    for col in cols:
        print(f"=== {col} vs target ===")
        summary = df.groupby(col, dropna=False)[target_col].agg(["mean", "count"])
        print(summary.round(3))
        print()


# -----------------------------
# 7. Basic leakage / sanity checks
# -----------------------------
def report_sanity_checks(train_df: pd.DataFrame, test_df: pd.DataFrame, id_col: str) -> None:
    print("=== Sanity checks ===")
    print(f"Duplicate rows in train (excluding id): {train_df.drop(columns=[id_col]).duplicated().sum()}")
    print(f"Duplicate id in train: {train_df[id_col].duplicated().sum()}")
    print(f"Overlap id between train and test: {len(set(train_df[id_col]) & set(test_df[id_col]))}")
    print()


# -----------------------------
# Run all EDA steps
# -----------------------------
report_target_balance(train_df, TARGET)
missing_vs_target_df = report_missing_vs_target(train_df, numeric_cols + categorical_cols, TARGET)
missing_corr_df = report_missing_cooccurrence(train_df, numeric_cols + categorical_cols)
numeric_dist_df = report_numeric_distribution(train_df, numeric_cols)
report_numeric_correlation(train_df, numeric_cols, TARGET)
report_categorical_vs_target(train_df, categorical_cols, TARGET)
report_sanity_checks(train_df, test_df, ID_COL)

### Findings

- **Target balance:** 70.9% / 29.1% — moderate imbalance, handled via
  stratified splitting rather than resampling.
- **Missingness is unrelated to the target:** `diff_vs_overall` is
  negligible (0.00001–0.004) across all 12 columns — no evidence of
  MNAR. This ruled out behavior-driven missingness (e.g. users hiding
  high usage) and supported leaving NaN untouched for tree-based models.
- **Top predictors (by correlation with target):** `daily_screen_time_hours`
  (0.611), `weekend_screen_time` (0.590), `social_media_hours` (0.532).
  `notifications_per_day` and `app_opens_per_day` show ~0 correlation.
- **Multicollinearity:** `daily_screen_time_hours` and `weekend_screen_time`
  are highly correlated (0.80) — expected to be handled well by tree-based
  models but noted for interpretation.
- **Categorical features are weak predictors:** target rate is nearly
  identical across all categories of `gender`, `stress_level`, and
  `academic_work_impact` (0.70–0.72).
- **No leakage or data quality issues:** zero duplicates, zero id overlap
  between train/test.

## 2. Missing Co-occurrence Deep Dive

The correlation matrix above showed two clusters of correlated
missingness: screen-time columns, and sleep/stress/academic columns.
This section checks whether missingness follows a strict "all-or-nothing"
pattern (suggesting survey skip-logic) or a weaker structural correlation
(more consistent with synthetic data generation artifacts).

In [ ]:
screen_time_cluster = [
    "daily_screen_time_hours", "social_media_hours",
    "gaming_hours", "weekend_screen_time",
]
sleep_stress_cluster = ["sleep_hours", "stress_level", "academic_work_impact"]


def report_cluster_missing_overlap(df: pd.DataFrame, cluster_cols: list[str], cluster_name: str) -> None:
    """
    Check how many columns within a cluster are missing simultaneously per row,
    and show the most common missingness combinations.
    """
    missing_mask = df[cluster_cols].isna()
    missing_count_per_row = missing_mask.sum(axis=1)

    print(f"=== {cluster_name}: number of columns missing simultaneously (per row) ===")
    dist = missing_count_per_row.value_counts().sort_index()
    dist_pct = (dist / len(df) * 100).round(3)
    print(pd.DataFrame({"row_count": dist, "pct": dist_pct}))
    print()

    # Show most common missing patterns as binary combinations, e.g. (True, False, True, False)
    pattern_counts = missing_mask.value_counts().head(10)
    print(f"=== {cluster_name}: top missingness patterns (True = missing) ===")
    print(pattern_counts)
    print()


report_cluster_missing_overlap(train_df, screen_time_cluster, "Screen-time cluster")
report_cluster_missing_overlap(train_df, sleep_stress_cluster, "Sleep/stress/academic cluster")

### Findings

- Only **0.95%** of rows are missing all 4 screen-time columns
  simultaneously — far below what a strict skip-logic pattern would
  produce, though ~12x higher than expected under full independence.
- The sleep/stress/academic cluster shows the same pattern (0.377% fully
  missing, ~11x higher than independence).
- **Conclusion:** missingness has a mild structural correlation but is
  not a hard "all-or-nothing" survey skip-logic — consistent with
  correlated noise introduced by the synthetic data generator rather
  than a meaningful behavioral signal. This confirmed the decision not
  to engineer a "missing cluster" indicator feature.